# Feature Engineering Combination v2

Jalankan setelah tiga notebook Extraction v1. Notebook ini hanya membaca 50.000 gambar training resmi dan checkpoint CNN v1. Kandidat A memakai embedding tersimpan; B merata-ratakan Dense dari citra asli dan flip; C merata-ratakan GlobalAveragePooling2D. Ketujuh kelompok fitur disimpan per kandidat di `outputs/cifar10_7fitur_v2/fusion/fitur/`. Jalankan semua cell berurutan. Tidak ada pelatihan CNN.

## 1. Konfigurasi dan identitas eksperimen

Seed, sumber v1, dan lokasi v2 ditetapkan sekali. Memory growth mencegah TensorFlow langsung menguasai seluruh VRAM.

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import os
import time

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 30092026
SOURCE_RUN = "cifar10_7fitur_v1"
RUN_ID = "cifar10_7fitur_v2"
ROOT = Path.cwd().resolve()
SOURCE = ROOT / "outputs" / SOURCE_RUN
OUT = ROOT / "outputs" / RUN_ID / "fusion"
BLOCK_NAMES = ("rgb", "avg", "ntsc", "hog", "lbp", "hsv", "rgb_stats")
VARIANTS = ("rgb", "avg", "ntsc")
REPRESENTATIONS = {"A": 512, "B": 512, "C": 256}
BATCH_SIZE = 128

for folder in ("model", "fitur", "laporan"):
    (OUT / folder).mkdir(parents=True, exist_ok=True)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("Output v2:", OUT)

I0000 00:00:1791496227.551461  225681 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791496227.590485  225681 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791496228.532893  225681 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Output v2: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v2/fusion


## 2. Data dan split pengembangan

Hanya bagian training CIFAR-10 yang dimuat. Indeks 40.000/10.000 harus sama persis dengan v1.

In [2]:
from tensorflow.keras.datasets import cifar10

(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.ravel().astype(np.int64)
if x_all.shape != (50000, 32, 32, 3):
    raise ValueError(f"Bentuk CIFAR-10 training tidak sesuai: {x_all.shape}")
train_idx, val_idx = train_test_split(
    np.arange(50000), test_size=0.2, stratify=y_all, random_state=SEED
)
train_idx, val_idx = np.sort(train_idx), np.sort(val_idx)
split_hash = hashlib.sha256(train_idx.tobytes() + val_idx.tobytes()).hexdigest()
print("Split:", len(train_idx), len(val_idx), split_hash)

Split: 40000 10000 16004a1ac8a89ac92ba6280367a7fb7b284c3643abb1c9d10409d6739ef31305


## 3. Periksa artefak sumber

Arsip gabungan dan masing-masing arsip CNN v1 harus cocok pada indeks, label, split, versi preprocessing, dimensi, serta SHA-256 checkpoint. Descriptor HOG/LBP/HSV/statistik diambil dari arsip gabungan yang lolos pemeriksaan ini.

In [3]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_path = SOURCE / "fusion" / "fitur" / "train_combined.npz"
if not source_path.exists():
    raise FileNotFoundError(f"Arsip v1 tidak ditemukan: {source_path}")
with np.load(source_path, allow_pickle=False) as archive:
    required = {"features", "y", "indices", "train_idx", "val_idx", "run_id",
                "split_hash", "preprocessing_version", "block_names", "boundaries",
                "model_hashes"}
    if not required.issubset(archive.files):
        raise ValueError(f"Metadata v1 tidak lengkap: {required - set(archive.files)}")
    expected_boundaries = np.array([0, 512, 1024, 1536, 1860, 2116, 2164, 2173])
    checks = [str(archive["run_id"]) == SOURCE_RUN,
              str(archive["split_hash"]) == split_hash,
              str(archive["preprocessing_version"]) == "v1",
              tuple(archive["block_names"].tolist()) == BLOCK_NAMES,
              np.array_equal(archive["boundaries"], expected_boundaries),
              np.array_equal(archive["indices"], np.arange(50000)),
              np.array_equal(archive["y"], y_all),
              np.array_equal(archive["train_idx"], train_idx),
              np.array_equal(archive["val_idx"], val_idx)]
    if not all(checks):
        raise ValueError("Metadata, indeks, atau label arsip gabungan v1 tidak cocok")
    original = archive["features"].astype(np.float32)
    model_hashes = archive["model_hashes"].tolist()
if original.shape != (50000, 2173) or not np.isfinite(original).all():
    raise ValueError("Fitur gabungan v1 tidak berhingga atau dimensinya salah")

model_paths = {}
for variant, expected_hash, start in zip(VARIANTS, model_hashes, (0, 512, 1024)):
    model_path = SOURCE / variant / "model" / "cnn.keras"
    archive_path = SOURCE / variant / "fitur" / "train_features.npz"
    if not model_path.exists() or not archive_path.exists():
        raise FileNotFoundError(f"Checkpoint atau arsip {variant} v1 tidak ditemukan")
    if sha256_file(model_path) != expected_hash:
        raise ValueError(f"Hash checkpoint {variant} berubah")
    with np.load(archive_path, allow_pickle=False) as archive:
        checks = [str(archive["run_id"]) == SOURCE_RUN,
                  str(archive["variant"]) == variant,
                  str(archive["model_sha"]) == expected_hash,
                  str(archive["split_hash"]) == split_hash,
                  str(archive["preprocessing_version"]) == "v1",
                  np.array_equal(archive["indices"], np.arange(50000)),
                  np.array_equal(archive["y"], y_all),
                  np.array_equal(archive["train_idx"], train_idx),
                  np.array_equal(archive["val_idx"], val_idx),
                  np.array_equal(archive["features"], original[:, start:start + 512])]
        if not all(checks):
            raise ValueError(f"Arsip {variant} tidak sejajar dengan gabungan v1")
    model_paths[variant] = model_path
print("Arsip v1 dan tiga checkpoint tervalidasi")

Arsip v1 dan tiga checkpoint tervalidasi


## 4. Rumus input CNN

RGB dibagi 255; AVG memakai rata-rata kanal; NTSC memakai bobot 0,299/0,587/0,114. Flip diterapkan setelah konversi, tanpa augmentasi acak.

In [4]:
def prepare_images(x, variant):
    x = np.asarray(x, dtype=np.float32) / 255.0
    if variant == "rgb":
        return x
    if variant == "avg":
        return np.mean(x, axis=-1, keepdims=True, dtype=np.float32)
    if variant == "ntsc":
        return np.sum(x * np.array([0.299, 0.587, 0.114], np.float32),
                      axis=-1, keepdims=True)
    raise ValueError(f"Varian tidak dikenal: {variant}")

## 5. Ekstraksi B dan C dari CNN beku

Setiap CNN dimuat satu per satu. Output Dense dan pooling diambil bersama dari citra asli dan flip dengan `training=False`, kemudian dirata-ratakan. Seluruh BatchNormalization tetap dalam mode inferensi.

In [5]:
def extract_flip_views(images, variant, model_path, batch_size=BATCH_SIZE):
    cnn = tf.keras.models.load_model(model_path, compile=False)
    dense_layer = cnn.get_layer("embedding")
    pool_layer = cnn.get_layer("global_average_pooling2d")
    if dense_layer.output.shape[-1] != 512 or pool_layer.output.shape[-1] != 256:
        raise ValueError(f"Dimensi layer CNN {variant} berubah")
    extractor = tf.keras.Model(cnn.input, [dense_layer.output, pool_layer.output])
    dense = np.empty((len(images), 512), dtype=np.float32)
    pool = np.empty((len(images), 256), dtype=np.float32)
    interval = max(1, int(np.ceil(len(images) / batch_size)) // 20)
    for batch_number, start in enumerate(range(0, len(images), batch_size), 1):
        end = min(start + batch_size, len(images))
        batch = prepare_images(images[start:end], variant)
        flipped = np.ascontiguousarray(batch[:, :, ::-1, :])
        dense_orig, pool_orig = extractor(batch, training=False)
        dense_flip, pool_flip = extractor(flipped, training=False)
        dense[start:end] = (dense_orig.numpy() + dense_flip.numpy()) * np.float32(0.5)
        pool[start:end] = (pool_orig.numpy() + pool_flip.numpy()) * np.float32(0.5)
        if batch_number % interval == 0 or end == len(images):
            print(f"{variant}: {end}/{len(images)}", flush=True)
    if not np.isfinite(dense).all() or not np.isfinite(pool).all():
        raise ValueError(f"Fitur flip {variant} mengandung NaN/inf")
    del extractor, cnn
    tf.keras.backend.clear_session()
    gc.collect()
    return dense, pool

## 6. Definisi arsip kandidat

Batas blok mengikuti dimensi layer CNN. Arsip disimpan secara atomik dan diperiksa kembali; arsip lama yang berbeda metadata ditolak.


In [6]:
def expected_boundaries_for(rep):
    dim = REPRESENTATIONS[rep]
    return np.cumsum([0, dim, dim, dim, 324, 256, 48, 9]).astype(np.int32)


def validate_candidate(path, rep):
    with np.load(path, allow_pickle=False) as archive:
        checks = [str(archive["run_id"]) == RUN_ID,
                  str(archive["representation"]) == rep,
                  str(archive["source_run"]) == SOURCE_RUN,
                  str(archive["split_hash"]) == split_hash,
                  str(archive["preprocessing_version"]) == "v2-group-l2-1",
                  tuple(archive["block_names"].tolist()) == BLOCK_NAMES,
                  np.array_equal(archive["boundaries"], expected_boundaries_for(rep)),
                  np.array_equal(archive["model_hashes"], model_hashes),
                  np.array_equal(archive["indices"], np.arange(50000)),
                  np.array_equal(archive["y"], y_all),
                  np.array_equal(archive["train_idx"], train_idx),
                  np.array_equal(archive["val_idx"], val_idx)]
        if not all(checks):
            raise ValueError(f"Arsip v2 {rep} tidak cocok; periksa provenance")
        feature = archive["features"]
        if feature.shape != (50000, int(expected_boundaries_for(rep)[-1])):
            raise ValueError(f"Dimensi kandidat {rep} salah: {feature.shape}")
        if not np.isfinite(feature).all():
            raise ValueError(f"Fitur kandidat {rep} mengandung NaN/inf")


def save_candidate(rep, feature):
    path = OUT / "fitur" / f"train_combined_{rep}.npz"
    if path.exists():
        validate_candidate(path, rep)
        print("Sudah valid, dilewati:", path.name)
        return
    boundaries = expected_boundaries_for(rep)
    if feature.shape != (50000, int(boundaries[-1])) or not np.isfinite(feature).all():
        raise ValueError(f"Fitur kandidat {rep} tidak valid")
    temporary = path.with_suffix(".npz.tmp")
    with open(temporary, "wb") as stream:
        np.savez_compressed(stream, features=feature, y=y_all,
                            indices=np.arange(50000), train_idx=train_idx,
                            val_idx=val_idx, run_id=RUN_ID,
                            representation=rep, source_run=SOURCE_RUN,
                            split_hash=split_hash,
                            preprocessing_version="v2-group-l2-1",
                            block_names=np.array(BLOCK_NAMES), boundaries=boundaries,
                            model_hashes=np.array(model_hashes))
    os.replace(temporary, path)
    validate_candidate(path, rep)
    print("Tersimpan:", path, feature.shape)



## 7. Simpan kandidat A

Kandidat asli memakai embedding Dense dan empat descriptor yang sudah tervalidasi di arsip v1.


In [7]:
combination_start = time.perf_counter()
for rep in REPRESENTATIONS:
    existing = OUT / "fitur" / f"train_combined_{rep}.npz"
    if existing.exists():
        validate_candidate(existing, rep)
        print("Sudah valid, dilewati:", existing.name)

if not (OUT / "fitur" / "train_combined_A.npz").exists():
    save_candidate("A", original)



Tersimpan: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v2/fusion/fitur/train_combined_A.npz (50000, 2173)


## 8. Ekstraksi B dan C

Jika salah satu arsip flip belum ada, inferensi asli dan flip dijalankan untuk tiap CNN secara bergantian. Hasil sementara ditahan di RAM.


In [8]:
need_flip = any(not (OUT / "fitur" / f"train_combined_{rep}.npz").exists()
                for rep in ("B", "C"))
if need_flip:
    four_descriptors = original[:, 1536:].copy()
    dense_parts, pool_parts = [], []
    for variant in VARIANTS:
        dense, pool = extract_flip_views(x_all, variant, model_paths[variant])
        dense_parts.append(dense)
        pool_parts.append(pool)
else:
    print("Kandidat B dan C sudah valid; ekstraksi dilewati")


I0000 00:00:1791496245.434667  225681 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4139 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1791496246.410774  225681 cuda_dnn.cc:461] Loaded cuDNN version 92600


rgb: 2432/50000
rgb: 4864/50000
rgb: 7296/50000
rgb: 9728/50000
rgb: 12160/50000
rgb: 14592/50000
rgb: 17024/50000
rgb: 19456/50000
rgb: 21888/50000
rgb: 24320/50000
rgb: 26752/50000
rgb: 29184/50000
rgb: 31616/50000
rgb: 34048/50000
rgb: 36480/50000
rgb: 38912/50000
rgb: 41344/50000
rgb: 43776/50000
rgb: 46208/50000
rgb: 48640/50000
rgb: 50000/50000
avg: 2432/50000
avg: 4864/50000
avg: 7296/50000
avg: 9728/50000
avg: 12160/50000
avg: 14592/50000
avg: 17024/50000
avg: 19456/50000
avg: 21888/50000
avg: 24320/50000
avg: 26752/50000
avg: 29184/50000
avg: 31616/50000
avg: 34048/50000
avg: 36480/50000
avg: 38912/50000
avg: 41344/50000
avg: 43776/50000
avg: 46208/50000
avg: 48640/50000
avg: 50000/50000
ntsc: 2432/50000
ntsc: 4864/50000
ntsc: 7296/50000
ntsc: 9728/50000
ntsc: 12160/50000
ntsc: 14592/50000
ntsc: 17024/50000
ntsc: 19456/50000
ntsc: 21888/50000
ntsc: 24320/50000
ntsc: 26752/50000
ntsc: 29184/50000
ntsc: 31616/50000
ntsc: 34048/50000
ntsc: 36480/50000
ntsc: 38912/50000
ntsc: 4134

## 9. Simpan kandidat B dan C

Gabungkan tiga blok CNN dengan empat descriptor v1, lalu periksa ulang bentuk dan metadata arsip.


In [9]:
if need_flip:
    if not (OUT / "fitur" / "train_combined_B.npz").exists():
        save_candidate("B", np.concatenate(dense_parts + [four_descriptors], axis=1))
    if not (OUT / "fitur" / "train_combined_C.npz").exists():
        save_candidate("C", np.concatenate(pool_parts + [four_descriptors], axis=1))
    del dense_parts, pool_parts, four_descriptors
    gc.collect()
print("Durasi kombinasi (detik):", round(time.perf_counter() - combination_start, 1))


Tersimpan: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v2/fusion/fitur/train_combined_B.npz (50000, 2173)
Tersimpan: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v2/fusion/fitur/train_combined_C.npz (50000, 1405)
Durasi kombinasi (detik): 166.1


## 10. Manifest provenance

Manifest menyimpan hash tiap arsip dan checkpoint. File v1 tidak ditulis ulang. Waktu Combination dicatat sebagai bagian anggaran komputasi v2.


In [10]:
manifest_path = OUT / "laporan" / "combination_manifest.json"
archive_hashes = {}
for rep in REPRESENTATIONS:
    path = OUT / "fitur" / f"train_combined_{rep}.npz"
    validate_candidate(path, rep)
    archive_hashes[rep] = sha256_file(path)
manifest = {
    "run_id": RUN_ID, "source_run": SOURCE_RUN, "seed": SEED,
    "split_hash": split_hash, "model_hashes": model_hashes,
    "source_archive_sha256": sha256_file(source_path),
    "archive_sha256": archive_hashes,
    "block_names": BLOCK_NAMES,
    "boundaries": {rep: expected_boundaries_for(rep).tolist()
                   for rep in REPRESENTATIONS},
    "preprocessing_version": "v2-group-l2-1",
    "combination_seconds": float(time.perf_counter() - combination_start),
    "inference_mode": "training=False; horizontal flip deterministic",
}
if manifest_path.exists():
    previous = json.loads(manifest_path.read_text(encoding="utf-8"))
    for key in ("run_id", "source_run", "seed", "split_hash", "model_hashes",
                "source_archive_sha256", "archive_sha256", "boundaries"):
        if previous[key] != manifest[key]:
            raise ValueError(f"Manifest lama berbeda pada {key}; jangan gabungkan eksperimen")
    manifest["combination_seconds"] = previous["combination_seconds"]
else:
    temporary = manifest_path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    os.replace(temporary, manifest_path)
print("Manifest:", manifest_path)
print("SHA arsip:", archive_hashes)

Manifest: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v2/fusion/laporan/combination_manifest.json
SHA arsip: {'A': '7fa1227d2b0db49b797e70442d852e98a504539f61f4ac31dde403331edc76f9', 'B': '913e80a3acc3508ef3fab0750b0df51bfbf885446149ca03766aa3027fa57fc8', 'C': 'c7c3616aa3e2d727197b66c17f6b179ca3d90ffb15408e7acc01ae341137bab4'}
